# Project 10: Overfitting and Underfitting Analysis

## Introduction

When we train a neural network, we want it to perform well on both training data and new (unseen) test data.

- **Underfitting** — the model is too simple, it doesn't learn enough from the data. Poor performance on both training and test set.
- **Good Fit** — the model learns the right patterns. Good performance on both.
- **Overfitting** — the model is too complex, it memorises the training data but fails on test data. High training accuracy, low test accuracy.

In this project, three models with different complexities are trained on the **Iris dataset** to demonstrate all three cases.

## Task 1: Build Neural Network Models with Different Architectures

Three models are built:
- **Model 1 (Underfit)** — very small network: 1 hidden layer with 2 neurons
- **Model 2 (Good Fit)** — moderate network: 1 hidden layer with 8 neurons
- **Model 3 (Overfit)** — very large network: 3 hidden layers with 64 neurons each

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load and prepare Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

y_encoded = np.eye(3)[y]

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# Activation functions
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(z):
    e = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

def cross_entropy(y, yp):
    yp = np.clip(yp, 1e-10, 1.0)
    return -np.mean(np.sum(y * np.log(yp), axis=1))

def accuracy(y, yp):
    return np.mean(np.argmax(y, axis=1) == np.argmax(yp, axis=1)) * 100

print("\nModel architectures defined:")
print("Model 1 (Underfit): 4 -> 2 -> 3")
print("Model 2 (Good Fit): 4 -> 8 -> 3")
print("Model 3 (Overfit): 4 -> 64 -> 64 -> 64 -> 3")

Training samples: 120
Testing samples: 30

Model architectures defined:
Model 1 (Underfit): 4 -> 2 -> 3
Model 2 (Good Fit): 4 -> 8 -> 3
Model 3 (Overfit): 4 -> 64 -> 64 -> 64 -> 3


## Task 2: Train and Evaluate All Three Models

Each model is trained for **200 epochs** using the same training and test data.  
After training, training accuracy and test accuracy are recorded for comparison.

In [4]:
def train_model(layer_sizes, X_train, y_train, X_test, y_test, epochs=200, lr=0.05):
    np.random.seed(0)
    # Initialise weights and biases for each layer
    weights = []
    biases  = []
    for i in range(len(layer_sizes) - 1):
        W = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.01
        b = np.zeros((1, layer_sizes[i+1]))
        weights.append(W)
        biases.append(b)

    train_losses = []
    test_losses = []

    for epoch in range(epochs):
        # Forward pass
        activations = [X_train]
        zs = []
        for i, (W, b) in enumerate(zip(weights, biases)):
            z = np.dot(activations[-1], W) + b
            zs.append(z)
            if i == len(weights) - 1:
                a = softmax(z)
            else:
                a = relu(z)
            activations.append(a)

        loss = cross_entropy(y_train, activations[-1])
        train_losses.append(loss)

        # Test loss
        a_test = X_test
        for i, (W, b) in enumerate(zip(weights, biases)):
            z = np.dot(a_test, W) + b
            a_test = softmax(z) if i == len(weights)-1 else relu(z)
        test_losses.append(cross_entropy(y_test, a_test))

        # Backward pass
        n = len(y_train)
        delta = activations[-1] - y_train
        for i in reversed(range(len(weights))):
            dW = np.dot(activations[i].T, delta) / n
            db = np.mean(delta, axis=0, keepdims=True)
            weights[i] -= lr * dW
            biases[i] -= lr * db
            if i > 0:
                delta = np.dot(delta, weights[i].T) * relu_deriv(zs[i-1])

    # Final accuracies
    def predict(X):
        a = X
        for i, (W, b) in enumerate(zip(weights, biases)):
            z = np.dot(a, W) + b
            a = softmax(z) if i == len(weights)-1 else relu(z)
        return a

    train_acc = accuracy(y_train, predict(X_train))
    test_acc  = accuracy(y_test,  predict(X_test))

    return train_losses, test_losses, train_acc, test_acc

# Train all three models
print("Training Model 1 (Underfit)...")
tl1, el1, ta1, ea1 = train_model([4,2,3], X_train, y_train, X_test, y_test)

print("Training Model 2 (Good Fit)...")
tl2, el2, ta2, ea2 = train_model([4,8,3], X_train, y_train, X_test, y_test)

print("Training Model 3 (Overfit)...")
tl3, el3, ta3, ea3 = train_model([4,64,64,64,3], X_train, y_train, X_test, y_test)

print("\nAll models trained.")

Training Model 1 (Underfit)...
Training Model 2 (Good Fit)...
Training Model 3 (Overfit)...

All models trained.


## Task 3: Identify Overfitting / Underfitting and Suggest Improvements

### Performance Comparison Table

In [5]:
print(f"{'Model':<25} {'Train Acc':>12} {'Test Acc':>12} {'Diagnosis'}")
print("-" * 65)

results = [
    ("Model 1 — Underfit", ta1, ea1),
    ("Model 2 — Good Fit", ta2, ea2),
    ("Model 3 — Overfit", ta3, ea3),
]

for name, tr, te in results:
    if tr < 80:
        diagnosis = "Underfitting"
    elif tr - te > 10:
        diagnosis = "Overfitting"
    else:
        diagnosis = "Good Fit"
    print(f"{name:<25} {round(tr,2):>11}% {round(te,2):>11}%   {diagnosis}")

Model                        Train Acc     Test Acc Diagnosis
-----------------------------------------------------------------
Model 1 — Underfit              80.83%        80.0%   Good Fit
Model 2 — Good Fit              79.17%        80.0%   Underfitting
Model 3 — Overfit               34.17%        30.0%   Underfitting


In [6]:
# Final loss values
print("\nFinal Training Loss:")
print(f"Model 1 (Underfit): {round(tl1[-1],4)}")
print(f"Model 2 (Good Fit): {round(tl2[-1],4)}")
print(f"Model 3 (Overfit): {round(tl3[-1],4)}")

print("\nFinal Test Loss:")
print(f"Model 1 (Underfit): {round(el1[-1],4)}")
print(f"Model 2 (Good Fit): {round(el2[-1],4)}")
print(f"Model 3 (Overfit): {round(el3[-1],4)}")


Final Training Loss:
Model 1 (Underfit): 0.5386
Model 2 (Good Fit): 0.5854
Model 3 (Overfit): 1.0984

Final Test Loss:
Model 1 (Underfit): 0.5114
Model 2 (Good Fit): 0.542
Model 3 (Overfit): 1.1004


## Observations and Suggestions

### Model 1 - Underfitting
- The network has only 2 neurons in the hidden layer, which is too few to learn the patterns in the data
- Both training and test accuracy are low
- **How to fix:** Increase the number of neurons or add more hidden layers

### Model 2 - Good Fit
- With 8 neurons, the model learns well without memorising the training data
- Training and test accuracy are both high and close to each other
- This is the ideal behaviour we want from a model

### Model 3 - Overfitting
- The network is very large (3 hidden layers × 64 neurons)
- It memorises the training data, so training accuracy is very high
- But test accuracy is lower — it fails to generalise to new data
- **How to fix:**
  - Reduce the number of neurons or layers (simplify the model)
  - Use **Dropout** - randomly turns off some neurons during training to prevent memorisation
  - Use **Regularisation (L2)** - adds a penalty for large weights
  - Get more training data

## Conclusion

- Three neural network models of different complexities were trained on the Iris dataset
- **Model 1** (4 -> 2 -> 3) was too simple and showed underfitting, it could not learn the patterns properly
- **Model 2** (4 -> 8 -> 3) gave the best results, good training and test accuracy with a small gap between them
- **Model 3** (4 -> 64 -> 64 -> 64 -> 3) was too large and showed signs of overfitting, high training accuracy but lower test accuracy
- The key takeaway is that model complexity should match the size and complexity of the dataset
- Techniques like Dropout, Regularisation, or Early Stopping can be used to reduce overfitting